# Investigation: account 35922 (2026-10-20)
Account `35922` appears in `bbg.verification_sched_detail` but is not one of the report accounts (Main 301491, PORT 791626). Goal: work out what it is, where it comes from, and whether it should be in or out of the report.

In [0]:
%run "../utils"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

ACC = "35922"
sd = tables("bbg.verification_sched_detail", "rob6332").current().withColumn("REPORT_YR_MONTH", F.date_format("PROCESSED_TIME", "yyyy/MM"))
ss = tables("bbg.verification_sched_summary", "rob6332").current()
log = tables("bbg.verification_file_process_log", "rob6332").current()

In [0]:
display(sd.groupBy("ACCOUNT_ID").agg(F.count("*").alias("rows"), F.countDistinct("ID_BB_UNIQUE").alias("securities"), F.min("PROCESSED_TIME").alias("first"), F.max("PROCESSED_TIME").alias("last")).orderBy("ACCOUNT_ID"))
display(ss.groupBy("ACCOUNT_ID").agg(F.count("*").alias("rows"), F.min("REPORT_YR_MONTH").alias("first"), F.max("REPORT_YR_MONTH").alias("last")).orderBy("ACCOUNT_ID"))

In [0]:
s = sd.where(col("ACCOUNT_ID") == ACC)
display(s.groupBy("REPORT_YR_MONTH", "SERVICE", "CTRB_BILLING").agg(F.count("*").alias("rows"), F.countDistinct("ID_BB_UNIQUE").alias("securities"), F.countDistinct("OUTPUT_ID").alias("outputs")).orderBy("REPORT_YR_MONTH", "SERVICE"))

In [0]:
display(s.groupBy("BILLABLE_PRODUCT", "DOWNLOAD_PRODUCT").agg(F.count("*").alias("rows"), F.countDistinct("ID_BB_UNIQUE").alias("securities")).orderBy(F.desc("rows")))
display(s.groupBy("OUTPUT_ID", "SERVICE").agg(F.count("*").alias("rows"), F.min("PROCESSED_TIME").alias("first"), F.max("PROCESSED_TIME").alias("last")).orderBy(F.desc("rows")).limit(50))

In [0]:
# The detail file is per account, so 35922 rows must come inside a Main or PORT file (or its own file).
# Compare row sample columns and the process log file names.
display(log.where(col("file_name").contains(ACC)))
display(log.select(F.regexp_extract("file_name", r"dlverreport_(\d+)_", 1).alias("file_account")).groupBy("file_account").count())
display(s.select("CTRB_AC", "CTRB_BAC", "BO_OPTIMIZED", "QUALIFIER", "FEATURES").groupBy("CTRB_AC", "CTRB_BAC", "BO_OPTIMIZED", "QUALIFIER").count().orderBy(F.desc("count")))

In [0]:
# Same securities and same timestamps on Main/PORT would suggest 35922 is a sub-account or contributor of a report account
k = ["ID_BB_UNIQUE", "BILLABLE_PRODUCT", "REPORT_YR_MONTH"]
x = s.select(*k).distinct()
for acc in ["301491", "791626"]:
    o = sd.where(col("ACCOUNT_ID") == acc).select(*k).distinct()
    print(acc, "overlap pairs:", x.join(o, k).count(), "of", x.count())
same_ts = s.alias("a").join(sd.where(col("ACCOUNT_ID") != ACC).alias("b"), ["ID_BB_UNIQUE", "PROCESSED_TIME"]).select("b.ACCOUNT_ID", "a.OUTPUT_ID", col("b.OUTPUT_ID").alias("other_output")).groupBy("ACCOUNT_ID", "OUTPUT_ID", "other_output").count().orderBy(F.desc("count"))
display(same_ts.limit(30))

#### Findings
Run on 2026-10-20 against the tables as loaded at that date.

* **Volume:** 2,960 detail rows / 1,481 securities, only **2024/01 and 2024/02**. Summary has 2,970 rows for the same two months.
* **Single request:** one `OUTPUT_ID`, `GLAC-EGF - MSCI World`, `SERVICE = P`, one timestamp per month (1st of the month, 01:00). Looks like a monthly MSCI World constituent pull for a fund/strategy.
* **Single product:** `Sched Equity SecMaster` only.
* **Not billed on 35922:** summary `TOTAL_USAGE = 0` and `GROUP_AC = 0` for every row, `NUM_OF_ACCOUNTS = 1`.
* **Fully overlaps Main:** 2,960 of 2,960 (security, product, month) pairs are also on Main 301491 in the same month; 0 on PORT.
* **How it got in:** files `dlverreport_35922_202401/202402` were loaded on 2025-04-02 during the initial backfill. `202403`-`202405` were set to `OVERRIDE` minutes later, as was account `787768`. `KNOWN_ACCOUNTS` now excludes both, so no new 35922 data will arrive.

**Conclusion:** a separate Bloomberg account with a single MSCI World universe request, no billed usage of its own, picked up by the early backfill before the account filter existed. Leftover data, not part of the invoice scope.

**Recommendation:** exclude it in `analysis_v1` by restricting both tables to `ACCOUNTS` at load. Optionally delete the 35922 rows from the tables (not done; destructive, needs your go-ahead). Who owns account 35922 is still open: ask the Bloomberg account manager or the team behind `GLAC-EGF`.